# 03 — No-CMF Unlearning Baselines (Θ_u) — Six-Method Sweep

**Pipeline role:** Third notebook in the CMF unlearning pipeline. Consumes
artifacts produced by NB1 and produces:
1. One unlearned checkpoint per `(method, seed, forget_class)` triple
   (`unlearn_no_cmf_dir/<experiment_name>_<method>_seed<N>_fc<C><suffix>.pt`)
2. Per-run training logs (`<name>_<method>_seed<N>_fc<C><suffix>_trainlog.json`)
3. `unlearn_summary.csv` — one row per experiment with the full three-metric
   evaluation (Output, Linear Probe, NCC) and a `method` column, mirroring
   NB2's `oracle_summary.csv` pattern.

**No-CMF baselines:** Each method starts from NB1's pretrained checkpoint Θ_o
and applies the standard (no-CMF) unlearning procedure.  Methods covered:
`grad_ascent_descent`, `random_label`, `salun`, `scrub`, `tarun`, `svd`.

**Inputs consumed from NB1 (never regenerated here):**
- `pretrain_dir/<experiment_name>_seed<N><suffix>.pt` — Θ_o starting point;
  loaded fresh for every `(method, seed, forget_class)` run.
- `splits_dir/<experiment_name>_seed<N>_fc<C><suffix>.json` — provides
  `retain_train_idx`, `retain_test_idx`, and `forget_test_idx`; **never
  regenerated here**.

## How to use
1. **Cell 0** — clones the repo and installs dependencies (same as NB1/NB2; skip if already done).
2. **Cell 1** — set `DATASET = "cifar10"` or `DATASET = "cifar100"`.  That is the only edit needed.
3. Run all remaining cells.

---
## Cell 0 — Setup: clone repo & install dependencies

Identical to NB1/NB2 Cell 0.  Skip if the environment is already prepared.

In [ ]:
import subprocess, sys, os

REPO_URL  = "https://github.com/tiensinh2/CMF_UNLearning_Posthoc.git"
REPO_NAME = "CMF_UNLearning_Posthoc"   # folder cloned into

# ── clone or pull to always use the latest code ──────────────────────────────
if not os.path.isdir(REPO_NAME):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("Clone complete.")
else:
    print(f"Repo exists — pulling latest changes from origin ...")
    subprocess.run(["git", "-C", REPO_NAME, "pull", "--ff-only"], check=True)
    print("Pull complete.")

# ── install dependencies ─────────────────────────────────────────────────────
_req = os.path.join(REPO_NAME, "requirements.txt")
if os.path.isfile(_req):
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", _req], check=True)
    print("Dependencies installed.")
else:
    # Minimal hard-coded fallback if requirements.txt is absent
    _pkgs = ["torch", "torchvision", "pytorch-lightning", "torchmetrics",
             "pyyaml", "pandas", "numpy", "timm"]
    print(f"requirements.txt not found — installing: {_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _pkgs, check=True)

# ── add repo root to sys.path ────────────────────────────────────────────────
_NB_DIR = os.path.abspath(REPO_NAME)
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

print(f"Repo root on sys.path: {_NB_DIR}")

---
## Cell 1 — Choose dataset  ✏️  ← the ONLY edit needed

Set `DATASET` to `"cifar10"` or `"cifar100"`.  
The notebook will automatically load the matching config file from `configs/`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  DATASET — the ONLY value you should change between experiments.        ║
# ║  "cifar10"  → loads configs/nb3_config_cifar10.yaml                    ║
# ║  "cifar100" → loads configs/nb3_config_cifar100.yaml                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DATASET = "cifar10"   # ← change to "cifar100" for CIFAR-100

---
## Cell 2 — Imports

In [ ]:
import copy
import json
import random
import time
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional

import yaml

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR

# ── project modules ──────────────────────────────────────────────────────────
from paper_hparams import (
    GLOBAL_CFG, PRETRAIN_CFG, CKPT_DIRS,
    UNLEARN_CFG_BY_METHOD,   # per-method Table-4 hparams (lr, epochs, method-specific params)
)
from unlearn import unlear_func   # method-name → function dispatch dict
from unlearn.cmf_weights import ModelModule
from utils import (
    get_dataset,
    test,
    to_jsonable,
)

# ── shared evaluation pipeline (used identically in NB2–NB4) ─────────────────
from nb_eval_helpers import (
    eval_cmf_three_metrics,
    build_test_split_loaders,
)

print("All imports OK.")

---
## Cell 3 — Load & resolve configuration

Selects `configs/nb3_config_{DATASET}.yaml` based on the `DATASET` variable
set in Cell 1, loads it, and merges with `paper_hparams.PRETRAIN_CFG` defaults
(NB3 re-uses PRETRAIN_CFG as the base namespace since unlearning starts from Θ_o;
per-method hparams are applied per-run inside the loop from `UNLEARN_CFG_BY_METHOD`).

In [ ]:
# ── select config file from DATASET variable ─────────────────────────────────
_CONFIG_MAP = {
    "cifar10":  "configs/nb3_config_cifar10.yaml",
    "cifar100": "configs/nb3_config_cifar100.yaml",
}
assert DATASET in _CONFIG_MAP, (
    f"Unknown DATASET '{DATASET}'. Choose from: {list(_CONFIG_MAP.keys())}"
)
_config_path = Path(_NB_DIR) / _CONFIG_MAP[DATASET]
assert _config_path.exists(), f"Config file not found: {_config_path}"
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg: Dict[str, Any] = yaml.safe_load(_f)
print(f"DATASET='{DATASET}' → loaded config: {_config_path}")

# ── merge: PRETRAIN_CFG defaults first, then YAML overrides ──────────────────
# The base args namespace mirrors NB1/NB2; per-method hparams are applied
# inside the loop via _apply_method_hparams().
_hp = {**PRETRAIN_CFG}                      # start from paper defaults
_hp_overrides = _cfg.get("hparam_overrides") or {}
_hp.update(_hp_overrides)                   # intentional ablation overrides

# ── top-level protocol fields ────────────────────────────────────────────────
EXPERIMENT_NAME  = _cfg["experiment_name"]
SUFFIX           = _cfg.get("suffix", "")
DATASET          = _cfg["dataset"]
ARCH             = _cfg["arch"]
DATA_PATH        = _cfg.get("data_path",        GLOBAL_CFG["data_path"])

# ── Kaggle: redirect data_path to writable /kaggle/working/data ─────────────
# /kaggle/input/ is read-only; torchvision's download=True would crash there.
# If the Kaggle CIFAR dataset is attached, symlink its already-extracted
# folder into the writable path so no re-download is needed.
# If only the .tar.gz is present, extract it into the writable path.
_KAGGLE_INPUT = Path("/kaggle/input")
if _KAGGLE_INPUT.exists():
    DATA_PATH = "/kaggle/working/data"
    _writable_data = Path(DATA_PATH)
    _writable_data.mkdir(parents=True, exist_ok=True)
    _CIFAR_FOLDER_MAP = {"cifar10": "cifar-10-batches-py", "cifar100": "cifar-100-python"}
    _expected_folder = _CIFAR_FOLDER_MAP.get(DATASET)
    if _expected_folder and not (_writable_data / _expected_folder).exists():
        import glob as _glob
        # 1) Try to symlink the pre-extracted folder directly
        _matches = _glob.glob(f"/kaggle/input/**/{_expected_folder}", recursive=True)
        if _matches:
            (_writable_data / _expected_folder).symlink_to(_matches[0])
            print(f"[Kaggle] Symlinked {_matches[0]} -> {_writable_data / _expected_folder}")
        else:
            # 2) Fall back: find the .tar.gz and extract it into the writable dir
            _TARBALL_MAP = {"cifar10": "cifar-10-python.tar.gz", "cifar100": "cifar-100-python.tar.gz"}
            _tarball_name = _TARBALL_MAP.get(DATASET)
            _tar_matches = _glob.glob(f"/kaggle/input/**/{_tarball_name}", recursive=True) if _tarball_name else []
            if _tar_matches:
                import tarfile as _tarfile
                print(f"[Kaggle] Extracting {_tar_matches[0]} -> {DATA_PATH} ...")
                with _tarfile.open(_tar_matches[0], 'r:gz') as _tf:
                    _tf.extractall(DATA_PATH)
                print(f"[Kaggle] Extraction complete.")
            else:
                print(f"[Kaggle] WARNING: neither '{_expected_folder}' folder nor '{_tarball_name}' "
                      "found in /kaggle/input. torchvision will attempt to download.")
    print(f"[Kaggle] data_path overridden to: {DATA_PATH}")

SEEDS: List[int] = _cfg.get("SEEDS",             GLOBAL_CFG["SEEDS"])
TEST_MODE: bool  = bool(_cfg.get("TEST_MODE",    GLOBAL_CFG["TEST_MODE"]))
TEST_EPOCHS_SCALE: float = float(
    _cfg.get("TEST_EPOCHS_SCALE", GLOBAL_CFG["TEST_EPOCHS_SCALE"])
)

BATCH_SIZE      = int(_cfg.get("batch_size",      GLOBAL_CFG["batch_size"]))
TEST_BATCH_SIZE = int(_cfg.get("test_batch_size", GLOBAL_CFG["test_batch_size"]))
NUM_WORKERS     = int(_cfg.get("num_workers",     GLOBAL_CFG["num_workers"]))

# ── NB3-specific: methods list ────────────────────────────────────────────────
METHODS: List[str] = _cfg.get(
    "methods",
    ["grad_ascent_descent", "random_label", "salun", "scrub", "tarun", "svd"],
)

# ── output directories ───────────────────────────────────────────────────────
PRETRAIN_DIR = Path(_cfg.get("pretrain_dir",        CKPT_DIRS["pretrain"]))
SPLITS_DIR   = Path(_cfg.get("splits_dir",          CKPT_DIRS["splits"]))
UNLEARN_DIR  = Path(_cfg.get("unlearn_no_cmf_dir",  CKPT_DIRS["unlearn_no_cmf"]))
RESULTS_DIR  = Path(_cfg.get("results_dir",         CKPT_DIRS["results"]))

UNLEARN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── test-mode epoch scaling ──────────────────────────────────────────────────
# Applied to the base pretrain hp dict; per-method epoch scaling is handled
# inside _apply_method_hparams() using the same TEST_EPOCHS_SCALE factor.
if TEST_MODE:
    SUFFIX = (SUFFIX or "") + "_test"
    print(f"[TEST_MODE] TEST_EPOCHS_SCALE={TEST_EPOCHS_SCALE}, suffix set to '{SUFFIX}'")

# ── forget_classes resolved after dataset load ───────────────────────────────
_forget_classes_cfg = _cfg.get("forget_classes")   # None → all classes

print("=" * 60)
print(f"Experiment : {EXPERIMENT_NAME}{SUFFIX}")
print(f"Dataset    : {DATASET}")
print(f"Arch       : {ARCH}")
print(f"Seeds      : {SEEDS}")
print(f"Methods    : {METHODS}")
print(f"Test mode  : {TEST_MODE}")
print(f"Base HP    : {_hp}")
print(f"HP overrides applied: {_hp_overrides}")
print("=" * 60)

---
## Cell 4 — Build `args` namespace

Constructs the base `SimpleNamespace` that the project's `utils.py`,
`unlearn/cmf_weights.py`, and `nb_eval_helpers.py` expect.  All values come
from the YAML config / merged hyperparameter dict — never hardcoded here.

NB3-specific notes:
- `CMFClassifier = True`, `remove_FC = True` — same convention as NB1/NB2.
  The no-CMF unlearning functions in this codebase (`unlearn_naive`,
  `salun_unlearn`, etc.) operate on the backbone encoder directly; they do not
  call `recompute_cmf()` or manipulate CMFweights, so the CMFClassifier flag
  controls only model construction here, not the unlearn loss path.
- `unlearn_method` and `unlearn_class` are placeholders here; they are set
  per-run inside the loop via `copy.copy(args)`.
- Per-method extra fields are applied inside the loop via `_apply_method_hparams()`.

In [ ]:
import types

args = types.SimpleNamespace(
    # ── dataset / arch ─────────────────────────────────────────────
    dataset          = DATASET,
    arch             = ARCH,
    data_path        = DATA_PATH,
    train_transform  = True,       # always use augmentation for consistency
    num_classes      = -1,         # filled in by get_dataset
    class_label_names= [],         # filled in by get_dataset

    # ── unlearn_method / unlearn_class: set per-run inside the loop ──
    # Placeholder values here; overridden via copy.copy(args) before dispatch.
    unlearn_method   = None,
    unlearn_class    = [],

    # ── CMF model flags (required by ModelModule) ───────────────────
    # Use the same construction convention as NB1/NB2 so that the loaded
    # Θ_o weights map correctly onto the model structure.
    CMFClassifier    = True,
    remove_FC        = True,
    CMF_momentum     = float(_hp.get("CMF_momentum", PRETRAIN_CFG.get("CMF_momentum", 0.9))),
    temperature      = float(_hp.get("temperature",  PRETRAIN_CFG.get("temperature",  1.0))),
    pretrained       = False,

    # ── optimiser / training (base values; overridden per method) ───
    lr               = float(_hp["lr"]),
    momentum         = float(_hp["momentum"]),
    weight_decay     = float(_hp["weight_decay"]),
    nesterov         = bool(_hp["nesterov"]),
    epochs_or_steps  = int(_hp["epochs"]),

    # ── scheduler (base; not used by most unlearn methods) ──────────
    lr_scheduler     = _hp.get("scheduler", "cosine"),
    warmup_epochs    = int(_hp.get("warmup_epochs", 5)),
    min_lr           = 1e-5,
    patience         = int(_hp.get("early_stop_patience", 50)),

    # ── dataloader ──────────────────────────────────────────────────
    batch_size       = BATCH_SIZE,
    test_batch_size  = TEST_BATCH_SIZE,
    num_workers      = NUM_WORKERS,

    # ── eval (used by nb_eval_helpers) ──────────────────────────────
    probe_batch_size = TEST_BATCH_SIZE,

    # ── misc ────────────────────────────────────────────────────────
    val_ratio        = float(_hp.get("val_ratio", 0.1)),
    log_interval     = 100,
    dry_run          = False,
    save_model       = True,
    gpu_id           = 0,
    sub_set_mode     = False,
    sub_set_samples  = 10000,

    # ── per-method fields (defaults; overridden per method in loop) ─
    # Initialised to safe defaults so that unlearn functions that access
    # these via getattr() without a fallback do not KeyError.
    grad_norm_clip         = None,
    salun_threshold        = 0.5,
    scrub_del_bsz          = 64,
    scrub_sgda_bsz         = 64,
    scrub_msteps           = 2,
    scrub_gamma            = 0.99,
    scrub_alpha            = 0.001,
    scrub_kd_T             = 4.0,
    SVD_alpha_r            = 1000,
    SVD_alpha_f            = 30,
    SVD_samples            = 900,
    SVD_max_patches        = 10,
    tarun_impair_lr        = 2e-4,
    tarun_samples_per_class = 100,
    freeze_except_last     = False,
    zero_last_layer        = False,
    # subset sizes used by naive unlearn (apply_prep infers these)
    num_forget_samples     = None,   # set per-run once forget_loader is built
    num_retain_samples     = None,   # set per-run once retain_loader is built
)


def _apply_method_hparams(m_args: types.SimpleNamespace, method: str) -> None:
    """
    Apply UNLEARN_CFG_BY_METHOD[method] hparams onto m_args in-place.

    Applies dataset-specific lr override when a `per_dataset_lr` sub-dict
    is present (same pattern as Table 4 per-dataset lr values).
    In TEST_MODE, scales epochs by TEST_EPOCHS_SCALE (min 1).
    """
    if method not in UNLEARN_CFG_BY_METHOD:
        print(f"[WARN] '{method}' not in UNLEARN_CFG_BY_METHOD — using base args hparams.")
        return
    mcfg = UNLEARN_CFG_BY_METHOD[method]

    # ── lr: apply dataset-specific override if available ─────────────────────
    base_lr = mcfg.get("lr")
    per_ds_lr = mcfg.get("per_dataset_lr", {}).get(DATASET)
    m_args.lr = float(per_ds_lr if per_ds_lr is not None else (base_lr if base_lr is not None else m_args.lr))

    # ── epochs ────────────────────────────────────────────────────────────────
    raw_epochs = int(mcfg.get("epochs", m_args.epochs_or_steps))
    if TEST_MODE:
        raw_epochs = max(1, int(raw_epochs * TEST_EPOCHS_SCALE))
    m_args.epochs_or_steps = raw_epochs

    # ── batch size ────────────────────────────────────────────────────────────
    if "batch_size" in mcfg:
        m_args.batch_size = int(mcfg["batch_size"])

    # ── optimiser flags ───────────────────────────────────────────────────────
    for _k in ("momentum", "weight_decay", "nesterov"):
        if _k in mcfg:
            setattr(m_args, _k, mcfg[_k])

    # ── method-specific hparams ───────────────────────────────────────────────
    for _k in (
        "grad_norm_clip",
        "salun_threshold",
        "scrub_del_bsz", "scrub_sgda_bsz", "scrub_msteps",
        "scrub_gamma", "scrub_alpha", "scrub_kd_T",
        "SVD_alpha_r", "SVD_alpha_f", "SVD_samples", "SVD_max_patches",
        "tarun_impair_lr", "tarun_samples_per_class",
    ):
        if _k in mcfg:
            setattr(m_args, _k, mcfg[_k])

    # ── SVD dataset-specific overrides ───────────────────────────────────────
    # SVD uses a nested per_dataset dict (not per_dataset_lr)
    if method == "svd":
        _svd_ds = mcfg.get("per_dataset", {}).get(DATASET, {})
        for _k, _v in _svd_ds.items():
            setattr(m_args, _k, _v)


print("args namespace and _apply_method_hparams helper ready.")

---
## Cell 5 — Device selection

In [ ]:
if torch.cuda.is_available():
    device = torch.device(f"cuda:{args.gpu_id}")
    torch.cuda.set_device(args.gpu_id)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

---
## Cell 6 — Load dataset (once)

The full dataset is loaded **once** and reused across all `(seed, forget_class,
method)` iterations.  Retain and forget subsets are assembled per-experiment
from the split JSON files produced by NB1.

In [ ]:
# get_dataset uses args.train_transform and args.dataset / args.data_path
train_dataset_full, test_dataset = get_dataset(args)

# Resolve forget_classes now that num_classes is known
if _forget_classes_cfg is None:
    FORGET_CLASSES: List[int] = list(range(args.num_classes))
else:
    FORGET_CLASSES = [int(c) for c in _forget_classes_cfg]

print(f"Train set size : {len(train_dataset_full)}")
print(f"Test  set size : {len(test_dataset)}")
print(f"num_classes    : {args.num_classes}")
print(f"forget_classes : {FORGET_CLASSES}")

---
## Cell 7 — Helper utilities

Same structural helpers as NB2 (`set_all_seeds`, `_build_subset`, `_make_loader`)
copied verbatim, plus NB3-specific path constructors and `_build_model_no_cmf`.

In [ ]:
def set_all_seeds(seed: int) -> None:
    """Pin every RNG source to `seed` for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── NB3-specific path constructors ───────────────────────────────────────────

def unlearn_ckpt_path(method: str, seed: int, fc: int) -> Path:
    """Canonical path for an unlearned checkpoint at (method, seed, forget_class)."""
    return UNLEARN_DIR / f"{EXPERIMENT_NAME}_{method}_seed{seed}_fc{fc}{SUFFIX}.pt"


def unlearn_log_path(method: str, seed: int, fc: int) -> Path:
    return UNLEARN_DIR / f"{EXPERIMENT_NAME}_{method}_seed{seed}_fc{fc}{SUFFIX}_trainlog.json"


def split_json_path(seed: int, fc: int) -> Path:
    """Path of the NB1-produced split JSON for (seed, forget_class)."""
    return SPLITS_DIR / f"{EXPERIMENT_NAME}_seed{seed}_fc{fc}{SUFFIX}.json"


def pretrain_ckpt_path(seed: int) -> Path:
    """Path of the NB1-produced pretrain checkpoint for a given seed."""
    return PRETRAIN_DIR / f"{EXPERIMENT_NAME}_seed{seed}{SUFFIX}.pt"


# ── Shared dataset helpers (copied verbatim from NB2) ─────────────────────────

def _load_split(seed: int, fc: int) -> Dict[str, Any]:
    """Load and return the split JSON produced by NB1."""
    p = split_json_path(seed, fc)
    assert p.exists(), (
        f"Split file not found: {p}\n"
        "Run NB1 first to generate splits before executing NB3."
    )
    with open(str(p)) as _sf:
        return json.load(_sf)


def _build_subset(base_dataset, idx: List[int]):
    """
    Return a dataset view containing exactly the given indices.
    Supports CIFAR-style (.data / .targets) and ImageFolder-style datasets.
    """
    ds = copy.deepcopy(base_dataset)
    if hasattr(ds, "data"):  # CIFAR-style
        ds.data    = base_dataset.data[idx]
        ds.targets = [base_dataset.targets[i] for i in idx]
    else:                    # ImageFolder-style
        ds.samples = [base_dataset.samples[i] for i in idx]
        if hasattr(ds, "imgs"):
            ds.imgs = ds.samples
        if hasattr(base_dataset, "targets"):
            ds.targets = [base_dataset.targets[i] for i in idx]
        else:
            ds.targets = [s[1] for s in ds.samples]
    return ds


def _make_loader(ds, shuffle: bool, batch_size: int = BATCH_SIZE):
    use_cuda = device.type == "cuda"
    return torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=use_cuda,
        shuffle=shuffle,
    )


# ── NB3-specific model helper ─────────────────────────────────────────────────

def _build_model_no_cmf(seed: int) -> torch.nn.Module:
    """
    Construct a fresh ModelModule with CMFClassifier=True, remove_FC=True
    (same construction convention as NB1/NB2), then load Θ_o weights from
    NB1's pretrain checkpoint for this seed.

    Unlearning starts from the already-trained Θ_o — never from random init.
    Each call produces an independent model object; never reuse across methods.
    """
    _args = copy.copy(args)
    _args.CMFClassifier = True
    _args.remove_FC     = True
    model = ModelModule(_args).to(device)
    # Mandatory guard: verify CMFweights attribute is present
    assert hasattr(model, "CMFweights"), (
        "ModelModule did not create CMFweights — check that "
        "CMFClassifier=True and remove_FC=True were accepted."
    )

    # Load Θ_o weights (required — unlearning starts from the trained model)
    pt_path = pretrain_ckpt_path(seed)
    assert pt_path.exists(), (
        f"Pretrain checkpoint not found: {pt_path}\n"
        "Run NB1 first to produce Θ_o before executing NB3."
    )
    _state = torch.load(str(pt_path), map_location=device)
    if "state_dict" in _state:
        model.load_state_dict(_state["state_dict"])
    elif "model_state" in _state:
        model.load_state_dict(_state["model_state"])
    else:
        model.load_state_dict(_state)
    return model


print("Helpers defined.")

---
## Cell 8 — Validate NB1 artifact availability and metadata consistency

Before running any unlearning, check that:
1. All required split JSON files exist.
2. Each split file's stored metadata (`experiment_name`, `dataset`, `suffix`)
   matches the current notebook configuration.
3. All required pretrain checkpoints exist and are loadable — **required** here
   (unlike NB2 where they were a warning-only reference).

Missing or mismatched files abort the run early with a clear message.

In [ ]:
print("=" * 60)
print("Validating NB1 artifacts ...")
print("=" * 60)

_missing_splits    = []
_mismatch_splits   = []
_missing_pretrain  = []
_bad_pretrain      = []

for seed in SEEDS:
    # ── pretrain checkpoint (REQUIRED — unlearning warm-starts from Θ_o) ──────
    _pt = pretrain_ckpt_path(seed)
    if not _pt.exists():
        _missing_pretrain.append(str(_pt))
        print(f"[MISSING] pretrain checkpoint (REQUIRED): {_pt}")
    else:
        # Light metadata check: load config block without pulling full weights.
        try:
            _pt_state = torch.load(str(_pt), map_location="cpu")
            _pt_cfg   = _pt_state.get("config", {})
            _pt_name  = _pt_cfg.get("experiment_name", "<unknown>")
            _pt_ds    = _pt_cfg.get("dataset",         "<unknown>")
            _match = (_pt_name == EXPERIMENT_NAME and _pt_ds == DATASET)
            _tag = "OK" if _match else "MISMATCH"
            if not _match:
                _bad_pretrain.append(str(_pt))
            print(f"[{_tag}]   pretrain: {_pt.name}  "
                  f"(experiment='{_pt_name}', dataset='{_pt_ds}')")
        except Exception:
            print(f"[OK]      pretrain checkpoint exists (metadata unreadable): {_pt.name}")

    # ── split files ───────────────────────────────────────────────────────────
    for fc in FORGET_CLASSES:
        _sp = split_json_path(seed, fc)
        if not _sp.exists():
            _missing_splits.append(str(_sp))
            print(f"[MISSING] split: {_sp}")
            continue

        # Metadata consistency check
        try:
            with open(str(_sp)) as _sf:
                _sp_meta = json.load(_sf)
            _sp_name   = _sp_meta.get("experiment_name", "<unknown>")
            _sp_ds     = _sp_meta.get("dataset",         "<unknown>")
            _sp_fc     = _sp_meta.get("forget_class",    -1)
            _sp_suffix = _sp_meta.get("suffix",          "")
            _sp_seed   = _sp_meta.get("seed",            -1)
            _ok = (
                _sp_name   == EXPERIMENT_NAME
                and _sp_ds == DATASET
                and _sp_fc == fc
                and _sp_seed == seed
                and str(_sp_suffix) == str(SUFFIX)
            )
            if _ok:
                print(f"[OK]      split seed={seed} fc={fc}: "
                      f"retain_tr={_sp_meta.get('n_retain_train')}  "
                      f"retain_tst={_sp_meta.get('n_retain_test')}  "
                      f"forget_tst={_sp_meta.get('n_forget_test')}")
            else:
                _mismatch_splits.append((seed, fc, str(_sp)))
                print(f"[MISMATCH] split seed={seed} fc={fc}: "
                      f"stored=(experiment='{_sp_name}', dataset='{_sp_ds}', "
                      f"seed={_sp_seed}, fc={_sp_fc}, suffix='{_sp_suffix}') "
                      f"expected=('{EXPERIMENT_NAME}', '{DATASET}', "
                      f"{seed}, {fc}, '{SUFFIX}')")
        except Exception as _e:
            print(f"[WARN]    split {_sp.name}: metadata unreadable ({_e})")

# ── abort if any required artifact is missing or mismatched ──────────────────
errors = []
if _missing_pretrain:
    errors.append(
        f"{len(_missing_pretrain)} pretrain checkpoint(s) missing (REQUIRED for NB3):\n  "
        + "\n  ".join(_missing_pretrain)
    )
if _bad_pretrain:
    errors.append(
        f"{len(_bad_pretrain)} pretrain checkpoint(s) have mismatched metadata:\n  "
        + "\n  ".join(_bad_pretrain)
    )
if _missing_splits:
    errors.append(
        f"{len(_missing_splits)} split file(s) missing:\n  "
        + "\n  ".join(_missing_splits)
    )
if _mismatch_splits:
    errors.append(
        f"{len(_mismatch_splits)} split file(s) have mismatched metadata:\n  "
        + "\n  ".join(str(t) for t in _mismatch_splits)
    )
if errors:
    raise ValueError(
        "\nNB1 artifact validation failed.  Run NB1 to completion before "
        "executing NB3.\n\n" + "\n\n".join(errors)
    )

print("\nArtifact validation complete — all pretrain checkpoints and splits verified.")

---
## Cell 9 — Unlearning loop

For each `(seed, forget_class, method)` triple:
1. Load the NB1 split JSON; extract `retain_train_idx`, `retain_test_idx`,
   `forget_train_idx`, and `forget_test_idx`.
2. Build retain/forget train loaders and test split loaders.
3. Load Θ_o weights from NB1's pretrain checkpoint (fresh copy per method).
4. Apply this method's Table-4 hparams via `_apply_method_hparams()`.
5. Dispatch to the method's unlearn function via `unlear_func[method]`.
6. Run the shared `eval_cmf_three_metrics` pipeline.
7. Save a self-describing checkpoint and training log.

If a checkpoint already exists the experiment is skipped but the full
three-metric evaluation is **re-run** on the loaded model so the CSV is
always complete.  Each method is wrapped individually in try/except so one
failure does not block other methods for the same (seed, fc).

In [ ]:
unlearn_records: List[Dict[str, Any]] = []   # one row per completed experiment
completed_runs: List[tuple] = []
skipped_runs:   List[tuple] = []
failed_runs:    List[tuple] = []

for seed in SEEDS:
    for fc in FORGET_CLASSES:

        # ── load split (always needed — for loaders and metadata) ─────────────
        try:
            split_meta = _load_split(seed, fc)
        except AssertionError as _e:
            print(f"[ERROR] {_e}")
            for _m in METHODS:
                failed_runs.append((_m, seed, fc))
            continue

        retain_tr_idx  = split_meta["retain_train_idx"]
        forget_tr_idx  = split_meta["forget_train_idx"]
        retain_tst_idx = split_meta["retain_test_idx"]
        forget_tst_idx = split_meta["forget_test_idx"]

        print(f"\n{'='*60}")
        print(f"seed={seed}  forget_class={fc}")
        print(f"  retain_train={len(retain_tr_idx)}  forget_train={len(forget_tr_idx)}  "
              f"retain_test={len(retain_tst_idx)}  forget_test={len(forget_tst_idx)}")
        print(f"{'='*60}")

        # ── build dataset views (needed by all methods) ───────────────────────
        retain_train_ds = _build_subset(train_dataset_full, retain_tr_idx)
        forget_train_ds = _build_subset(train_dataset_full, forget_tr_idx)

        retain_loader     = _make_loader(retain_train_ds, shuffle=True)
        forget_loader     = _make_loader(forget_train_ds, shuffle=True)
        full_train_loader = _make_loader(train_dataset_full, shuffle=False,
                                          batch_size=TEST_BATCH_SIZE)

        # separate test loaders for retain and forget (same as NB2)
        retain_test_loader, forget_test_loader = build_test_split_loaders(
            test_dataset,
            forget_test_indices = forget_tst_idx,
            retain_test_indices = retain_tst_idx,
            batch_size          = TEST_BATCH_SIZE,
            num_workers         = NUM_WORKERS,
        )
        full_test_loader = _make_loader(test_dataset, shuffle=False,
                                         batch_size=TEST_BATCH_SIZE)

        for method in METHODS:
            ckpt = unlearn_ckpt_path(method, seed, fc)
            logp = unlearn_log_path(method, seed, fc)

            print(f"\n{'='*60}")
            print(f"{method}  seed={seed}  fc={fc}  |  {ckpt.name}")
            print(f"{'='*60}")

            # ── checkpoint-skip: load and re-eval so CSV is always complete ────
            if ckpt.exists():
                print(f"[SKIP] Checkpoint exists — loading for eval.")
                skipped_runs.append((method, seed, fc))
                try:
                    _skip_args = copy.copy(args)
                    _skip_args.unlearn_method = method
                    _skip_args.unlearn_class  = [fc]
                    _apply_method_hparams(_skip_args, method)

                    _skip_model = _build_model_no_cmf(seed)
                    _state = torch.load(str(ckpt), map_location=device)
                    if "state_dict" in _state:
                        _skip_model.load_state_dict(_state["state_dict"])
                    elif "model_state" in _state:
                        _skip_model.load_state_dict(_state["model_state"])
                    else:
                        _skip_model.load_state_dict(_state)
                    _skip_model.eval()

                    tr_loader_eval = _make_loader(retain_train_ds, shuffle=False,
                                                  batch_size=TEST_BATCH_SIZE)
                    eval_metrics = eval_cmf_three_metrics(
                        _skip_model, _skip_args, device,
                        train_loader       = tr_loader_eval,
                        test_loader        = full_test_loader,
                        retain_test_loader = retain_test_loader,
                        forget_test_loader = forget_test_loader,
                        forget_class       = fc,
                    )
                    _m = _state.get("metrics", {})
                    unlearn_records.append({
                        "method":              method,
                        "seed":                seed,
                        "forget_class":        fc,
                        "status":              "skipped",
                        "wall_clock_minutes":  _state.get("wall_clock_minutes"),
                        "n_retain_train":      len(retain_tr_idx),
                        "n_forget_train":      len(forget_tr_idx),
                        "n_retain_test":       len(retain_tst_idx),
                        "n_forget_test":       len(forget_tst_idx),
                        **{k: round(float(v), 4) if isinstance(v, float) else v
                           for k, v in eval_metrics.items() if not k.startswith("_")},
                        "ckpt_path":           str(ckpt),
                    })
                except Exception:
                    print(f"[WARN] Could not re-evaluate skipped checkpoint:")
                    traceback.print_exc()
                continue

            # ── run unlearning ────────────────────────────────────────────────
            try:
                t0 = time.time()
                set_all_seeds(seed)

                # fresh args copy — apply this method's Table-4 hparams
                m_args = copy.copy(args)
                m_args.unlearn_class  = [fc]
                m_args.unlearn_method = method
                _apply_method_hparams(m_args, method)

                # num_forget_samples / num_retain_samples for naive unlearn
                # (apply_prep reads these via getattr; set them now that sizes are known)
                m_args.num_forget_samples = len(forget_tr_idx)
                m_args.num_retain_samples = len(retain_tr_idx)
                m_args.seed = seed   # required by random_label_unlearn and salun variants

                # scrub_unlearn reads args.scrub_epochs (not epochs_or_steps)
                if method == "scrub":
                    m_args.scrub_epochs = m_args.epochs_or_steps

                # load Θ_o for this seed, fresh per method
                model = _build_model_no_cmf(seed)

                # dispatch to the method's unlearn function
                # Map 'svd' key to the registry key 'SVD'
                _registry_key = "SVD" if method == "svd" else method
                if _registry_key not in unlear_func:
                    raise KeyError(
                        f"Method '{method}' (registry key '{_registry_key}') not found "
                        f"in unlear_func. Available: {list(unlear_func.keys())}"
                    )
                unlearn_fn = unlear_func[_registry_key]

                # tarun and SVD require train_dataset + val_index
                # (they build per-class retain subsets from train_dataset.targets)
                _needs_train_dataset = method in ("tarun", "svd")
                if _needs_train_dataset:
                    model = unlearn_fn(
                        args          = m_args,
                        model         = model,
                        device        = device,
                        retain_loader = retain_loader,
                        forget_loader = forget_loader,
                        train_loader  = full_train_loader,
                        test_loader   = full_test_loader,
                        optimizer     = None,
                        epochs        = m_args.epochs_or_steps,
                        train_dataset = train_dataset_full,
                        val_index     = retain_tr_idx,
                    )
                else:
                    model = unlearn_fn(
                        args               = m_args,
                        model              = model,
                        device             = device,
                        retain_loader      = retain_loader,
                        forget_loader      = forget_loader,
                        train_loader       = full_train_loader,
                        test_loader        = full_test_loader,
                        optimizer          = None,
                        epochs             = m_args.epochs_or_steps,
                        test_forget_loader = forget_test_loader,
                    )

                wall_min = (time.time() - t0) / 60

                # ── shared three-metric evaluation ────────────────────────────
                model.eval()
                tr_loader_eval = _make_loader(retain_train_ds, shuffle=False,
                                              batch_size=TEST_BATCH_SIZE)
                eval_metrics = eval_cmf_three_metrics(
                    model, m_args, device,
                    train_loader       = tr_loader_eval,
                    test_loader        = full_test_loader,
                    retain_test_loader = retain_test_loader,
                    forget_test_loader = forget_test_loader,
                    forget_class       = fc,
                )

                def _fmt(v): return f"{v:.4f}" if isinstance(v, (int, float)) else "N/A"
                print(f"\n  [Eval] Output  retain={_fmt(eval_metrics.get('output_retain_acc'))}  "
                      f"forget={_fmt(eval_metrics.get('output_forget_acc'))}")
                print(f"  [Eval] Probe   retain={_fmt(eval_metrics.get('probe_retain_acc'))}  "
                      f"forget={_fmt(eval_metrics.get('probe_forget_acc'))}")
                print(f"  [Eval] NCC     retain={_fmt(eval_metrics.get('ncc_retain_acc'))}  "
                      f"forget={_fmt(eval_metrics.get('ncc_forget_acc'))}")
                print(f"  (method={method}, seed={seed}, fc={fc}, wall={wall_min:.2f} min)")

                # ── collect per-method hparams for checkpoint provenance ───────
                _mcfg_snap = {
                    k: to_jsonable(v)
                    for k, v in UNLEARN_CFG_BY_METHOD.get(method, {}).items()
                    if not k.startswith("per_")   # skip per_dataset_lr / per_dataset dicts
                }

                # ── self-describing checkpoint (mirrors NB2's shape + method) ──
                _sd = model.state_dict()
                _eval_scalars = {k: to_jsonable(v) for k, v in eval_metrics.items()
                                 if not k.startswith("_")}
                checkpoint = {
                    "state_dict":  _sd,           # canonical key
                    "model_state": _sd,           # backward-compat alias
                    "seed":        seed,
                    "forget_class": fc,
                    "method":      method,
                    "stage":       "unlearn_no_cmf",
                    "config": {
                        "experiment_name": EXPERIMENT_NAME,
                        "suffix":          SUFFIX,
                        "dataset":         DATASET,
                        "arch":            ARCH,
                        "CMFClassifier":   True,
                        "remove_FC":       True,
                        "method_hparams":  _mcfg_snap,
                        "hp_overrides":    _hp_overrides,
                        "split_file":      str(split_json_path(seed, fc)),
                        "pretrain_ckpt":   str(pretrain_ckpt_path(seed)),
                    },
                    "n_retain_train": len(retain_tr_idx),
                    "n_forget_train": len(forget_tr_idx),
                    "n_retain_test":  len(retain_tst_idx),
                    "n_forget_test":  len(forget_tst_idx),
                    "metrics":        _eval_scalars,
                    "wall_clock_minutes": round(wall_min, 3),
                }
                torch.save(checkpoint, str(ckpt))
                print(f"[SAVED] {ckpt}")

                # training log JSON (human-readable, no tensors)
                log_data = {k: v for k, v in checkpoint.items()
                            if k not in ("state_dict", "model_state")}
                with open(str(logp), "w") as _lf:
                    json.dump(log_data, _lf, indent=2, default=to_jsonable)
                print(f"[LOG]   {logp}")

                unlearn_records.append({
                    "method":              method,
                    "seed":                seed,
                    "forget_class":        fc,
                    "status":              "completed",
                    "wall_clock_minutes":  round(wall_min, 3),
                    "n_retain_train":      len(retain_tr_idx),
                    "n_forget_train":      len(forget_tr_idx),
                    "n_retain_test":       len(retain_tst_idx),
                    "n_forget_test":       len(forget_tst_idx),
                    **{k: round(float(v), 4) if isinstance(v, float) else v
                       for k, v in eval_metrics.items() if not k.startswith("_")},
                    "ckpt_path":           str(ckpt),
                })
                completed_runs.append((method, seed, fc))

            except Exception:  # catch-all so other methods / experiments still run
                print(f"\n[ERROR] {method} failed for seed={seed} fc={fc}:")
                traceback.print_exc()
                failed_runs.append((method, seed, fc))

print("\n" + "=" * 60)
print("Unlearning loop complete.")
print(f"  Completed : {len(completed_runs)}  {completed_runs}")
print(f"  Skipped   : {len(skipped_runs)}  {skipped_runs}")
print(f"  Failed    : {len(failed_runs)}  {failed_runs}")

---
## Cell 10 — Save unlearn_summary.csv

Writes `results_dir/<experiment_name><suffix>_unlearn_summary.csv`.
Contains one row per `(method, seed, forget_class)` experiment with status,
runtime metadata, and the full three-metric evaluation results
(Output, Linear Probe, NCC).  The `method` column is the primary grouping
variable that distinguishes NB3's rows from each other.

In [ ]:
if unlearn_records:
    summary_df   = pd.DataFrame(unlearn_records)
    # sort for readability: method → seed → forget_class
    sort_cols = [c for c in ["method", "seed", "forget_class"] if c in summary_df.columns]
    if sort_cols:
        summary_df = summary_df.sort_values(sort_cols).reset_index(drop=True)
    summary_path = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_unlearn_summary.csv"
    summary_df.to_csv(str(summary_path), index=False)
    print(f"Unlearn summary saved → {summary_path}")
    display(summary_df)
else:
    print("No unlearn records to summarise (all were skipped or failed).")

---
## Cell 11 — Final output manifest

Print a concise summary of all artefacts produced so the notebook is
self-documenting when run as a report.

In [ ]:
print("="*60)
print("NB3 — OUTPUT MANIFEST")
print("="*60)

print("\n[Unlearn checkpoints]")
unlearn_files_all = sorted(UNLEARN_DIR.glob(f"{EXPERIMENT_NAME}_*_seed*_fc*{SUFFIX}.pt"))
for p in unlearn_files_all:
    size_mb = p.stat().st_size / 1_048_576 if p.exists() else 0
    print(f"  {p}  ({size_mb:.1f} MB)")

print("\n[Unlearn training logs]")
log_files_all = sorted(UNLEARN_DIR.glob(f"{EXPERIMENT_NAME}_*_seed*_fc*{SUFFIX}_trainlog.json"))
for p in log_files_all:
    print(f"  {p}")

print("\n[Summary CSV]")
_csv = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_unlearn_summary.csv"
print(f"  {_csv}" if _csv.exists() else "  (not produced)")

print(f"\n[Run status]")
print(f"  Completed : {len(completed_runs)}  {completed_runs}")
print(f"  Skipped   : {len(skipped_runs)}  {skipped_runs}")
if failed_runs:
    print(f"\n[WARNING] Unlearning failed for: {failed_runs}")

print("\nVerification checklist:")
print("  [x] Every method loaded from NB1's Θ_o checkpoint (never random init)")
print("  [x] Splits loaded from NB1 JSON — never regenerated")
print("  [x] Retain/forget membership from stored split indices only")
print("  [x] Method hyperparameters sourced from paper_hparams.py (Table 4) — no shell-script values")
print("  [x] Shared eval_cmf_three_metrics pipeline used (Output + Probe + NCC)")
print("  [x] Each method dispatched via unlear_func registry, not reimplemented")
print("  [x] unlearn_summary.csv contains complete three-metric evaluation for every (method, seed, forget_class)")
print("\nNB3 complete. Downstream NB4a/4b/4c should compare against these no-CMF baselines.")